# Customer Classification Model
**Dataset:** German Credit Data — UCI Machine Learning Repository  
**Obiettivo:** Classificare clienti bancari come *Good* (0) o *Bad* (1) in base a indicatori finanziari e anagrafici.  
**Stack:** Python · Scikit-learn · XGBoost · SMOTE · Streamlit

---
## Indice
1. [Caricamento e preprocessing](#1-caricamento)
2. [Analisi esplorativa (EDA)](#2-eda)
3. [Feature Engineering](#3-feature-engineering)
4. [Training modelli](#4-training)
5. [Confronto curve ROC](#5-roc)
6. [Bilanciamento con SMOTE](#6-smote)
7. [Feature Importance e salvataggio](#7-feature-importance)

## 1. Caricamento e Preprocessing
Caricamento diretto da UCI. I codici `A11`, `A43`, ecc. vengono mappati in valori leggibili tramite dizionario.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Caricamento dataset da UCI
url  = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
cols = ["checking","duration","credit_history","purpose","credit_amount",
        "savings","employment","installment_rate","personal_status",
        "guarantors","residence","property","age","other_installments",
        "housing","existing_credits","job","dependents","phone","foreign","target"]

df = pd.read_csv(url, sep=" ", header=None, names=cols)

# Target: 1=good → 0, 2=bad → 1
df["target"] = df["target"].map({1: 0, 2: 1})

# Mapping codici UCI → valori leggibili
mappings = {
    "checking":           {"A11": "<0 DM", "A12": "0-200 DM", "A13": ">200 DM", "A14": "no account"},
    "savings":            {"A61": "<100 DM", "A62": "100-500 DM", "A63": "500-1000 DM", "A64": ">1000 DM", "A65": "unknown/none"},
    "employment":         {"A71": "unemployed", "A72": "<1 year", "A73": "1-4 years", "A74": "4-7 years", "A75": ">7 years"},
    "housing":            {"A151": "rent", "A152": "own", "A153": "free"},
    "job":                {"A171": "unskilled non-resident", "A172": "unskilled resident", "A173": "skilled", "A174": "highly skilled"},
    "foreign":            {"A201": "foreign", "A202": "non-foreign"},
    "credit_history":     {"A30": "no credits taken", "A31": "all credits paid back duly",
                           "A32": "existing credits paid back duly", "A33": "delay in paying off", "A34": "critical account"},
    "purpose":            {"A40": "car (new)", "A41": "car (used)", "A42": "furniture/equipment",
                           "A43": "radio/TV", "A44": "domestic appliances", "A45": "repairs",
                           "A46": "education", "A47": "vacation", "A48": "retraining",
                           "A49": "business", "A410": "others"},
    "personal_status":    {"A91": "male divorced", "A92": "female divorced/married",
                           "A93": "male single", "A94": "male married", "A95": "female single"},
    "guarantors":         {"A101": "none", "A102": "co-applicant", "A103": "guarantor"},
    "property":           {"A121": "real estate", "A122": "savings/life insurance",
                           "A123": "car/other", "A124": "unknown/none"},
    "other_installments": {"A141": "bank", "A142": "stores", "A143": "none"},
    "phone":              {"A191": "no", "A192": "yes"},
}

for col, mapping in mappings.items():
    df[col] = df[col].map(mapping)

# Verifica
print("Shape:", df.shape)
print("\nTarget:\n", df["target"].value_counts())
print("\nValori mancanti:\n", df.isnull().sum())
df.head()

## 2. Analisi Esplorativa (EDA)
Visualizziamo la distribuzione delle variabili più rilevanti separando clienti *Good* e *Bad*.  
Le feature più interessanti da osservare sono: `checking`, `savings`, `duration`, `credit_history`.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Distribuzione variabili chiave per target", fontsize=14, fontweight="bold")

# Età
axes[0,0].hist(df[df["target"]==0]["age"], alpha=0.6, label="Good (0)", color="steelblue", bins=20)
axes[0,0].hist(df[df["target"]==1]["age"], alpha=0.6, label="Bad (1)",  color="tomato",    bins=20)
axes[0,0].set_title("Età")
axes[0,0].legend()

# Importo credito
axes[0,1].hist(df[df["target"]==0]["credit_amount"], alpha=0.6, label="Good (0)", color="steelblue", bins=20)
axes[0,1].hist(df[df["target"]==1]["credit_amount"], alpha=0.6, label="Bad (1)",  color="tomato",    bins=20)
axes[0,1].set_title("Importo credito")
axes[0,1].legend()

# Durata
axes[0,2].hist(df[df["target"]==0]["duration"], alpha=0.6, label="Good (0)", color="steelblue", bins=20)
axes[0,2].hist(df[df["target"]==1]["duration"], alpha=0.6, label="Bad (1)",  color="tomato",    bins=20)
axes[0,2].set_title("Durata (mesi)")
axes[0,2].legend()

# Conto corrente
checking_order = ["<0 DM", "0-200 DM", ">200 DM", "no account"]
ct_checking = df.groupby(["checking","target"]).size().unstack().reindex(checking_order)
ct_checking.plot(kind="bar", ax=axes[1,0], color=["steelblue","tomato"], alpha=0.8)
axes[1,0].set_title("Conto corrente")
axes[1,0].tick_params(axis="x", rotation=30)

# Risparmi
savings_order = ["<100 DM","100-500 DM","500-1000 DM",">1000 DM","unknown/none"]
ct_savings = df.groupby(["savings","target"]).size().unstack().reindex(savings_order)
ct_savings.plot(kind="bar", ax=axes[1,1], color=["steelblue","tomato"], alpha=0.8)
axes[1,1].set_title("Risparmi")
axes[1,1].tick_params(axis="x", rotation=30)

# Storia creditizia
ct_history = df.groupby(["credit_history","target"]).size().unstack()
ct_history.plot(kind="bar", ax=axes[1,2], color=["steelblue","tomato"], alpha=0.8)
axes[1,2].set_title("Storia creditizia")
axes[1,2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()

### Correlazioni e distribuzione importo credito

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Correlazioni e variabili numeriche", fontsize=14, fontweight="bold")

# Heatmap correlazione
num_cols = ["duration", "credit_amount", "installment_rate", "residence", "age", "existing_credits", "dependents", "target"]
corr = df[num_cols].corr()

sns.heatmap(
    corr, ax=axes[0], annot=True, fmt=".2f",
    cmap="coolwarm", center=0, square=True,
    linewidths=0.5, cbar_kws={"shrink": 0.8}
)
axes[0].set_title("Heatmap correlazione")
axes[0].tick_params(axis="x", rotation=45)

# Boxplot importo credito per target
axes[1].boxplot(
    [df[df["target"]==0]["credit_amount"], df[df["target"]==1]["credit_amount"]],
    tick_labels=["Good (0)", "Bad (1)"],
    patch_artist=True,
    boxprops=dict(facecolor="steelblue", alpha=0.6),
    medianprops=dict(color="red", linewidth=2)
)
axes[1].set_title("Importo credito per target")
axes[1].set_ylabel("Credit Amount (DM)")

plt.tight_layout()
plt.savefig("eda_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Feature Engineering
- **Label Encoding** sulle variabili categoriche
- **StandardScaler** sulle variabili numeriche continue
- Split X / y

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

df_model = df.copy()

# Encoding variabili categoriche
cat_cols = df_model.select_dtypes(include="str").columns.tolist()
print("Colonne encodate:", cat_cols)

le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

# Split X / y
X = df_model.drop(columns=["target"])
y = df_model["target"]

# Scaling variabili numeriche
num_cols = ["duration", "credit_amount", "installment_rate", "residence", "age", "existing_credits", "dependents"]
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

print(f"\nShape X: {X.shape} | Shape y: {y.shape}")
print("\nDistribuzione target:\n", y.value_counts())
X.head()

## 4. Training Modelli
Confronto tra tre modelli: **Logistic Regression**, **Random Forest**, **XGBoost**.  
Ogni modello viene valutato con AUC su test set e cross-validation a 5 fold.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
import warnings
warnings.filterwarnings("ignore")

# Split train/test (80/20) stratificato
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} righe | Test: {X_test.shape[0]} righe")

# Definizione modelli
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost":             XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss"),
}

# Training e valutazione
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    auc    = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    cv_auc = cross_val_score(model, X, y, cv=5, scoring="roc_auc").mean()
    results[name] = {"model": model, "auc": auc, "cv_auc": cv_auc}
    print(f"\n── {name} ──")
    print(classification_report(y_test, y_pred, target_names=["Good","Bad"]))
    print(f"AUC test: {auc:.3f} | AUC cross-val (5-fold): {cv_auc:.3f}")

## 5. Confronto Curve ROC
La curva ROC misura il trade-off tra True Positive Rate e False Positive Rate.  
**Random Forest** (senza SMOTE) ottiene l'AUC più alto: **0.795**.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, res in results.items():
    RocCurveDisplay.from_estimator(
        res["model"], X_test, y_test,
        ax=ax, name=f"{name} (AUC={res['auc']:.3f})"
    )

ax.plot([0,1],[0,1], "k--", label="Random baseline")
ax.set_title("Confronto curve ROC")
ax.legend(loc="lower right")
plt.savefig("roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Bilanciamento con SMOTE
Il dataset è sbilanciato (70% Good / 30% Bad). SMOTE genera esempi sintetici della classe minoritaria sul solo training set.  
Obiettivo: migliorare il **recall sui Bad** — in ambito bancario identificare i clienti a rischio è prioritario rispetto a minimizzare i falsi allarmi.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Prima di SMOTE:", y_train.value_counts().to_dict())
print("Dopo SMOTE:    ", dict(zip(*np.unique(y_train_sm, return_counts=True))))

# Riallena i modelli con SMOTE
results_sm = {}
for name, res in results.items():
    model = res["model"]
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test)
    auc    = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    results_sm[name] = {"model": model, "auc": auc}
    print(f"\n── {name} (con SMOTE) ──")
    print(classification_report(y_test, y_pred, target_names=["Good","Bad"]))
    print(f"AUC test: {auc:.3f}")

## 7. Feature Importance e Salvataggio Modelli
Salviamo entrambi i modelli:
- **XGBoost + SMOTE** → ottimizzato per il recall sui Bad (contesto bancario)
- **Random Forest** → ottimizzato per AUC massimo

Entrambi vengono salvati in `src/` insieme allo scaler per il deploy Streamlit.

In [ ]:
import joblib
import os

os.makedirs("../src", exist_ok=True)

best_xgb = results_sm["XGBoost"]["model"]
best_rf  = results["Random Forest"]["model"]

joblib.dump(best_xgb, "../src/model_xgb.pkl")
joblib.dump(best_rf,  "../src/model_rf.pkl")
joblib.dump(scaler,   "../src/scaler.pkl")

print("XGBoost salvato in src/model_xgb.pkl")
print("Random Forest salvato in src/model_rf.pkl")
print("Scaler salvato in src/scaler.pkl")

# Feature importance a confronto
feat_imp_xgb = pd.Series(best_xgb.feature_importances_, index=X.columns).sort_values(ascending=True)
feat_imp_rf  = pd.Series(best_rf.feature_importances_,  index=X.columns).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

feat_imp_xgb.plot(kind="barh", ax=axes[0], color="steelblue", alpha=0.8)
axes[0].set_title("Feature Importance — XGBoost (con SMOTE)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Importance")
axes[0].axvline(x=feat_imp_xgb.mean(), color="tomato", linestyle="--", label="Media")
axes[0].legend()

feat_imp_rf.plot(kind="barh", ax=axes[1], color="darkorange", alpha=0.8)
axes[1].set_title("Feature Importance — Random Forest (senza SMOTE)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Importance")
axes[1].axvline(x=feat_imp_rf.mean(), color="tomato", linestyle="--", label="Media")
axes[1].legend()

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()